# LLM 3v3 basketball — live tryout

Run the multi-agent game from this notebook and watch the JSONL log appear line by line.

- Keep `USE_FAKE = True` for an offline check (instant). Set it to `False` for real OpenRouter calls.
- Launch from the repo root so the package imports resolve:
  `uv run --with jupyterlab jupyter lab` (or open in VS Code and pick the `.venv` interpreter).
- The environment cell loads `.env` itself, so `--env-file` is optional once the notebook is running.
- Free tier: about 1,000 requests/day at about 20/min. A 6-event game is about 36 calls; 40 events is about 240.
- Fake and heuristic numbers are plumbing checks only, never M2 evidence (IMPLEMENTATION_PLAN.md §0 Decision 9).

In [1]:
import os
from pathlib import Path

REPO = Path.cwd()
while REPO.name != "llm-3v3-basketball-agents" and REPO.parent != REPO:
    REPO = REPO.parent
os.chdir(REPO)

env_path = REPO / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            name, _, value = line.partition("=")
            os.environ.setdefault(name.strip(), value.strip())

key = os.environ.get("OPENROUTER_API_KEY", "")
if key:
    print("OPENROUTER_API_KEY: present,", len(key), "chars")
else:
    print("OPENROUTER_API_KEY: MISSING - live calls will fail (fake mode still works)")
print("cwd:", Path.cwd())

OPENROUTER_API_KEY: present, 73 chars
cwd: c:\Users\hongy\OneDrive\Documents\Desktop\llm-3v3-basketball-agents


In [2]:
import asyncio
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from bball3x3.analysis import read_game_log, render_possession_transcript
from bball3x3.config import load_llm_config
from bball3x3.llm import FakeLLM, LegalXMLPolicy, OpenRouterClient
from bball3x3.runner.llm_game import run_llm_game

LLM_CFG = load_llm_config()

# Genuine calls by default; set enabled True to reuse cached replies instantly.
LLM_CFG.runtime.cache = {"enabled": False}

print("model:", LLM_CFG.players.model)
print("rpm limit:", LLM_CFG.runtime.max_requests_per_minute)

model: minimax/minimax-m3.1-flash:free
rpm limit: 18


In [3]:
USE_FAKE = False  # set False for real LLM calls

if USE_FAKE:
    CLIENT = FakeLLM(LegalXMLPolicy())
    print("backend: FakeLLM (offline, instant) - set USE_FAKE = False for live calls")
else:
    CLIENT = OpenRouterClient(LLM_CFG)
    print("backend:", LLM_CFG.players.model)

LOG = Path("runs/nb_llm_try.jsonl")


def _compact(rec: dict) -> str:
    kind = rec.get("record_type", "?")
    e = rec.get("event_idx")
    pre = f"E{e} " if e is not None else ""
    if kind == "game_start":
        return f"[game_start] model={rec.get('model_slug')} seed={rec.get('seed')}"
    if kind == "game_end":
        return f"[game_end] {json.dumps(rec.get('totals') or {}, default=str)[:160]}"
    if kind == "decision":
        tgt = f" {rec['executed_target']}" if rec.get("executed_target") else ""
        return (f"[decision] {pre}{rec.get('player')} -> "
                f"{rec.get('executed_action')}{tgt} parse_ok={rec.get('parse_ok')}")
    if kind == "message":
        d = rec.get("directive") or {}
        act = (f" [action to={d.get('to')} act={d.get('act')} "
               f"target={d.get('target')}]") if d else ""
        ch = " <c>" if rec.get("is_captain_channel") else ""
        return (f"[message] {pre}{rec.get('sender')}{ch} "
                f"({rec.get('kind')}): {rec.get('text')}{act}")
    if kind == "captain_call":
        tactic = rec.get("tactic") or {}
        if isinstance(tactic, dict):
            text = tactic.get("text") or ""
        else:
            text = getattr(tactic, "text", "") or ""
        return (f"[captain_call] {pre}team={rec.get('team')} "
                f"maintain={rec.get('maintain')} tactic={text!r} "
                f"directives={len(rec.get('directives') or [])}")
    scalars = {k: v for k, v in rec.items()
               if k not in ("record_type", "event_idx") and v is not None}
    return f"[{kind}] {pre}{json.dumps(scalars, default=str)[:200]}"


def _read_records(path=None):
    path = path or LOG
    if not path.exists():
        return []
    out = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            out.append(json.loads(line))
        except json.JSONDecodeError:
            break  # half-written trailing record: retry next tick, never skip
    return out


def _drain(pos):
    recs = _read_records()
    for rec in recs[pos:]:
        print(_compact(rec), flush=True)
    return len(recs)


async def run_live(max_events=6, seed=11, resume=False):
    if LOG.exists() and not resume:
        LOG.unlink()  # EventLogger refuses to overwrite an existing log
    task = asyncio.create_task(
        run_llm_game(llm=CLIENT, path=LOG, seed=seed,
                     max_events=max_events, resume=resume)
    )
    pos = 0
    while not task.done():
        pos = _drain(pos)
        await asyncio.sleep(1.0)
    _drain(pos)
    result = await task  # re-raises QuotaExceededError and friends
    print("\n--- game finished ---")
    return result


def replay_log(path=None):
    """Re-print an existing log through the current printer (no new game, no quota)."""
    recs = _read_records(path)
    if not recs:
        print("no log yet - run the game cell first")
        return
    for rec in recs:
        print(_compact(rec))


backend: minimax/minimax-m3.1-flash:free


In [4]:
MAX_EVENTS = 3

try:
    result = await run_live(max_events=MAX_EVENTS)
except Exception as exc:
    print(f"run stopped: {type(exc).__name__}: {exc}")
    print("if this was a daily-quota abort, rerun later via the resume cell")
    result = None

[game_start] model=minimax/minimax-m3.1-flash:free seed=11
[decision] E1 A1 -> DEFEND B1 parse_ok=True
[decision] E1 A2 -> HELP_PAINT parse_ok=True
[decision] E1 A3 -> HELP_PAINT parse_ok=True
[decision] E1 B1 -> DRIVE RIGHT parse_ok=True
[decision] E1 B2 -> MOVE LEFT_WING parse_ok=True
[decision] E1 B3 -> SCREEN A1 parse_ok=True
[captain_call] E1 team=A maintain=False tactic='' directives=0
[message] E1 A1 (tactic): contain the ball and protect the paint
[message] E1 A2 (info): help on the drive and protect the basket
[message] E1 A3 (info): Help at the lane; deny the interior pass.
[message] E1 B1 (action): cut the right block for a pass [action to=B3 act=CUT target=RIGHT_BLOCK]
[captain_call] E1 team=B maintain=False tactic='' directives=0
[message] E1 B1 (tactic): drive right and create a passing lane to the block
[message] E1 B2 (info): Moving to LEFT_WING for spacing and a potential open look.
[message] E1 B3 (action): screen for the drive [action to=B1 act=SCREEN target=A1]
[ref

In [ ]:
recs = [
    json.loads(line)
    for line in LOG.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
decisions = [r for r in recs if r.get("record_type") == "decision"]

df = pd.DataFrame(
    [
        {
            "event": r.get("event_idx"),
            "player": r.get("player"),
            "executed": r.get("executed_action"),
            "target": r.get("executed_target"),
            "parse_ok": r.get("parse_ok"),
            "invalid_reason": r.get("invalid_reason"),
            "latency_ms": round(r.get("latency_ms") or 0),
            "tokens": (r.get("tokens") or {}).get("total"),
        }
        for r in decisions
    ]
)
display(df)

n = len(df)
if n:
    parse_fail = (~df["parse_ok"].astype(bool)).mean()
    iar = df["invalid_reason"].notna().mean()
    print(f"decisions={n}  parse_fail={parse_fail:.2%} (target < 2%)  IAR={iar:.2%} (target < 5%)")
else:
    print("no decisions logged yet - run the game cell first")

,event,player,executed,target,parse_ok,invalid_reason,latency_ms,tokens
0,1,A1,DEFEND,B1,True,None,3005,2520
1,1,A2,HELP_PAINT,NaN,True,None,3282,2286
2,1,A3,HELP_PAINT,NaN,True,None,3528,2278
3,1,B1,DRIVE,RIGHT,True,None,6437,2655
4,1,B2,MOVE,LEFT_WING,True,None,2981,2215
5,1,B3,SCREEN,A1,True,None,3185,2250
6,2,A1,CONTEST,B1,True,None,2811,2659
7,2,A2,HELP_PAINT,NaN,True,None,2494,2352
8,2,A3,HELP_PAINT,NaN,True,None,2830,2349
9,2,B1,DRIVE,RIGHT,True,None,3976,2671


decisions=18  parse_fail=0.00% (target < 2%)  IAR=0.00% (target < 5%)


In [6]:
if LOG.exists():
    print(render_possession_transcript(read_game_log(LOG), "p0001"))
else:
    print("no log yet - run the game cell first")

Game llm_game_11 | possession p0001 | team B
Game log: unfinished | possession: open | events 1–3
Conditions: A=A_full, B=A_full
Boundary events may be shared with adjacent possessions.
Decisions belong to possession_before; scoring entries retain their possession IDs.
Recorded observations are authoritative; missing observation content is not inferred.

Event 1
Decision A1 (team A)
  Observation:
    [OWN STATE]
    Player: A1
    Team: A
    Role: DEFENSE
    Position: (7, 9)
    Has Ball: false
    Fatigue: 0.0/10
    
    [TEAMMATES]
    A2: pos=(3, 7), movement=HOLD, speed=NONE
    A3: pos=(11, 7), movement=HOLD, speed=NONE
    
    [OPPONENTS]
    B1: pos=(7, 6), movement=HOLD, speed=NONE, HAS_BALL
    B2: pos=(3, 5), movement=HOLD, speed=NONE
    B3: pos=(11, 5), movement=HOLD, speed=NONE
    
    [DEFENSIVE MATCHUPS]
    A1 -> B1
    A2 -> B2
    A3 -> B3
    B1 -> A1
    B2 -> A2
    B3 -> A3
    
    [BALL]
    Position: (7, 6)
    Status: HELD
    Holder: B1
    Possession: 

In [7]:
# Resume an aborted live run from its checkpoint (runs/nb_llm_try.ckpt.json):
# result = await run_live(max_events=MAX_EVENTS, resume=True)

if "recs" in globals():
    raw = [
        r.get("raw_output")
        for r in recs
        if r.get("record_type") == "decision" and r.get("raw_output")
    ]
    if raw:
        print("last raw model output:")
        print(raw[-1])
else:
    print("run the game and summary cells first")

last raw model output:
<RESPONSE>
  <ACTION type="ROLL" target="RIGHT_BLOCK"/>
  <MESSAGES>
    <B3>[action to=B1 act=CUT_BASELINE target=RIGHT_BLOCK] cut baseline to the right block for the pass[/action]</B3>
  </MESSAGES>
  <RATIONALE>Roll toward the right block to receive B1’s pass after the right-side drive.</RATIONALE>
</RESPONSE>


## Notes

- **Cache:** disabled in this notebook. If you re-enable it, repeated runs return cached replies instantly; clear `.cache/llm_cache.sqlite` before any run you intend to measure.
- **Quota:** free tier is about 1,000 requests/day at about 20/min; the client limiter is set to 18 RPM. The runner checkpoints per event, so a quota abort resumes with `resume=True`.
- **Truncation:** with `max_tokens: 1000` and low reasoning effort, `finish_reason=length` shows up as a cluster of PARSE_FAIL decisions, not an obvious error. Inspect raw outputs in the resume cell.
- **Baselines:** never point the log at `runs/game_101.jsonl` or `runs/game_11.jsonl`; those are heuristic baselines.
- **Standing:** fake/heuristic numbers are plumbing checks only; real-model numbers are required for M2 (IMPLEMENTATION_PLAN.md §0 Decision 9).